In [4]:
import pandas as pd
import numpy as np
import toad

print("正在读取真实的 Kaggle 数据集...")
# 1. 读取数据
df = pd.read_csv('cs-training.csv')
target = 'SeriousDlqin2yrs'

# 2. 真实数据特有的清洗步骤（必做！）
# 删掉第一列多余的索引（原版数据常常自带 Unnamed: 0）
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# 用中位数填充缺失值（不然 toad 跑分箱会报错）
df = df.fillna(df.median())

print(f"数据加载成功，共有 {len(df)} 条样本。")

# 3. 卡方分箱
print("\n正在跑卡方分箱...")
combiner = toad.transform.Combiner()
combiner.fit(df, y=target, method='chi', n_bins=5) 
binned_df = combiner.transform(df)

# 4. 计算 WOE 和 IV 值
transer = toad.transform.WOETransformer()
df_woe = transer.fit_transform(binned_df, df[target], exclude=[target])

# 5. 输出真实的 IV 值表
iv_table = toad.quality(df_woe, target, iv_only=True)
print("\n=== 🎯 真实特征的 IV 值计算结果 ===")
print(iv_table)

正在读取真实的 Kaggle 数据集...
数据加载成功，共有 150000 条样本。

正在跑卡方分箱...

=== 🎯 真实特征的 IV 值计算结果 ===
                                            iv  gini  entropy  unique
RevolvingUtilizationOfUnsecuredLines  1.100160   NaN      NaN     5.0
NumberOfTimes90DaysLate               0.877923   NaN      NaN     5.0
NumberOfTime30-59DaysPastDueNotWorse  0.759122   NaN      NaN     5.0
NumberOfTime60-89DaysPastDueNotWorse  0.600238   NaN      NaN     5.0
age                                   0.253054   NaN      NaN     5.0
NumberOfOpenCreditLinesAndLoans       0.095929   NaN      NaN     5.0
MonthlyIncome                         0.082904   NaN      NaN     5.0
DebtRatio                             0.069221   NaN      NaN     5.0
NumberRealEstateLoansOrLines          0.063401   NaN      NaN     5.0
NumberOfDependents                    0.034446   NaN      NaN     5.0


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

print("正在划分数据集并训练逻辑回归模型...")

# 1. 准备特征和标签（使用刚才算出来的真实特征）
features = iv_table.index.tolist() 
X = df_woe[features]
y = df_woe[target]

# 2. 划分训练集（70%）和测试集（30%）
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 3. 构建逻辑回归模型 (真实数据样本量大，增加 max_iter)
lr = LogisticRegression(class_weight='balanced', max_iter=2000)
lr.fit(X_train, y_train)

# 4. 对测试集进行预测（输出违约概率）
y_pred_proba = lr.predict_proba(X_test)[:, 1]

# 5. 计算 AUC 值
auc = roc_auc_score(y_test, y_pred_proba)
print(f"✅ 真实数据模型 AUC 值: {auc:.4f}")

# 6. 计算 KS 值
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
ks = max(tpr - fpr)
print(f"✅ 真实数据模型 KS 值: {ks:.4f}")

# 7. 将概率转化为标准信用评分
def prob_to_score(p, base=600, pdo=50):
    p = np.clip(p, 0.0001, 0.9999)
    odds = (1 - p) / p
    return base + (pdo / np.log(2)) * np.log(odds)

print("\n=== 信用分转化结果示例 ===")
sample_probas = y_pred_proba[:5]
for i, p in enumerate(sample_probas):
    score = prob_to_score(p)
    print(f"样本 {i+1}: 违约概率 {p:.4f} -> 标准信用评分 {score:.1f} 分")

正在划分数据集并训练逻辑回归模型...
✅ 真实数据模型 AUC 值: 0.8603
✅ 真实数据模型 KS 值: 0.5693

=== 信用分转化结果示例 ===
样本 1: 违约概率 0.1222 -> 标准信用评分 742.2 分
样本 2: 违约概率 0.1222 -> 标准信用评分 742.2 分
样本 3: 违约概率 0.1691 -> 标准信用评分 714.9 分
样本 4: 违约概率 0.6871 -> 标准信用评分 543.3 分
样本 5: 违约概率 0.5507 -> 标准信用评分 585.3 分


In [6]:
# 1. 定义计算 PSI 的函数
def calculate_psi(expected, actual, buckets=10):
    expected = np.where(expected == 0, 0.0001, expected)
    actual = np.where(actual == 0, 0.0001, actual)
    
    breakpoints = np.arange(0, buckets + 1) / buckets * 100
    breakpoints = np.percentile(expected, breakpoints)
    
    expected_percents = np.histogram(expected, breakpoints)[0] / len(expected)
    actual_percents = np.histogram(actual, breakpoints)[0] / len(actual)
    
    psi_value = 0
    for i in range(len(expected_percents)):
        e_perc = expected_percents[i] if expected_percents[i] > 0 else 0.0001
        a_perc = actual_percents[i] if actual_percents[i] > 0 else 0.0001
        psi_value += (e_perc - a_perc) * np.log(e_perc / a_perc)
        
    return psi_value

print("正在进行模型稳定性（PSI）监控...")

# 2. 算出训练集（基准月）和测试集（监控月）的所有分数
score_train = np.array([prob_to_score(p) for p in lr.predict_proba(X_train)[:, 1]])
score_test = np.array([prob_to_score(p) for p in lr.predict_proba(X_test)[:, 1]])

# 3. 计算 PSI
psi_value = calculate_psi(score_train, score_test)

# 4. 输出告警逻辑
print("\n=== 🚨 模型稳定性监控报表 ===")
print(f"基准样本量: {len(score_train)} | 当前样本量: {len(score_test)}")
print(f"整体模型 PSI 值: {psi_value:.4f}")
if psi_value < 0.1:
    print("✅ 状态: 模型稳定 (PSI < 0.1)，无需干预。")
elif psi_value <= 0.25:
    print("⚠️ 状态: 轻微偏移 (0.1 <= PSI <= 0.25)，需关注特征分布。")
else:
    print("🚨 告警: 严重偏移 (PSI > 0.25)，模型失效风险高，建议重新训练！")

# 5. 生成简单的监控报表
report_data = {
    '监控期': ['基准期 (Train)', '监控期 (Test)'],
    '样本量': [len(score_train), len(score_test)],
    '平均信用分': [round(np.mean(score_train), 1), round(np.mean(score_test), 1)],
    'PSI值': ['--', round(psi_value, 4)],
    '状态': ['基准', '正常' if psi_value < 0.1 else '需关注']
}
report_df = pd.DataFrame(report_data)
print("\n=== 自动化监控报表 ===")
print(report_df)

正在进行模型稳定性（PSI）监控...

=== 🚨 模型稳定性监控报表 ===
基准样本量: 105000 | 当前样本量: 45000
整体模型 PSI 值: 0.0002
✅ 状态: 模型稳定 (PSI < 0.1)，无需干预。

=== 自动化监控报表 ===
           监控期     样本量  平均信用分    PSI值  状态
0  基准期 (Train)  105000  655.2      --  基准
1   监控期 (Test)   45000  655.8  0.0002  正常
